# Text-to-Outfit — Diagnóstico del espacio de embeddings congelado

Objetivo: comprobar si `outfit_emb` (la salida congelada de `encode_outfit`) tiene geometría
suficiente para discriminar outfits por estilo, antes de invertir en fine-tuning. Ligero: solo
necesita `t2o_samples_with_embeddings.pt` del Notebook 1 (sin imágenes, sin modelo, sin GPU).


In [ ]:
import torch
import numpy as np
import random
from collections import defaultdict
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

T2O_CACHE_PATH = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/t2o_samples_with_embeddings.pt"  # ajusta si hace falta
t2o_samples = torch.load(T2O_CACHE_PATH)

embeddings = torch.stack([s["outfit_emb"] for s in t2o_samples])  # (N, 128), ya L2-normalizado
texts = [s["text"] for s in t2o_samples]
N = embeddings.shape[0]
print(f"Total de outfits cargados: {N}")

## 1. Similitud coseno entre pares de outfits aleatorios

Como los embeddings ya están L2-normalizados, el producto punto es directamente la similitud
coseno. Esta es la línea base "sin ninguna información" — la distribución de fondo del espacio.

**Cómo leerlo:** si la media está muy por encima de 0 (ej. > 0.4-0.5) y la desviación estándar
es pequeña, es indicio de colapso/anisotropía: casi todos los outfits están amontonados en la
misma región del espacio, independientemente de su contenido.

In [ ]:
n_pairs = 30000
idx_a = np.random.randint(0, N, n_pairs)
idx_b = np.random.randint(0, N, n_pairs)
mask = idx_a != idx_b
idx_a, idx_b = idx_a[mask], idx_b[mask]

sims_random = (embeddings[idx_a] * embeddings[idx_b]).sum(dim=1).numpy()

print(f"Similitud coseno entre pares ALEATORIOS (N={len(sims_random)}):")
print(f"  media={sims_random.mean():.3f}  std={sims_random.std():.3f}")
print(f"  percentiles -> p5={np.percentile(sims_random,5):.3f}  "
      f"p50={np.percentile(sims_random,50):.3f}  p95={np.percentile(sims_random,95):.3f}")

plt.figure(figsize=(7,4))
plt.hist(sims_random, bins=80, alpha=0.8)
plt.axvline(sims_random.mean(), color="red", linestyle="--", label=f"media={sims_random.mean():.3f}")
plt.title("Similitud coseno entre outfits aleatorios (línea base)")
plt.xlabel("Similitud coseno")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

## 2. ¿Los outfits con la MISMA etiqueta de estilo están más cerca entre sí que el azar?

Esta es la prueba más directa y la que más nos interesa. Tomamos las etiquetas de texto que se
repiten muchas veces (recordemos: 92,4% de los outfits comparten texto con otro), y comparamos
la similitud media **dentro** de cada etiqueta contra la línea base aleatoria de la celda anterior.

**Cómo leerlo:** si `sim_media_mismo_texto` no es claramente mayor que `sims_random.mean()`
para las etiquetas de estilo genuinas (`street style`, `casual`, `boho`, `formal`...), significa
que el espacio congelado no lleva señal de estilo -- ninguna cantidad de fine-tuning del
encoder de texto podría inventar una separación que no existe en el embedding de origen. Si SÍ
hay una diferencia clara, es una señal alentadora de que hay algo que extraer (aunque sea débil),
y ahí el fine-tuning del Transformer sí tendría sentido para amplificarlo.

In [ ]:
text_to_indices = defaultdict(list)
for i, t in enumerate(texts):
    text_to_indices[t].append(i)

MIN_REPEATS = 30
frequent_labels = [t for t, idxs in text_to_indices.items() if len(idxs) >= MIN_REPEATS]
frequent_labels = sorted(frequent_labels, key=lambda l: -len(text_to_indices[l]))
print(f"Etiquetas con >= {MIN_REPEATS} repeticiones: {len(frequent_labels)}")

def sample_same_label_pairs(label, n=300):
    idxs = text_to_indices[label]
    pairs = [tuple(random.sample(idxs, 2)) for _ in range(n)]
    return pairs

print(f"\n{'Etiqueta':30s} {'n outfits':>10s} {'sim. mismo texto':>18s} {'sim. aleatoria':>16s} {'diferencia':>12s}")
same_label_results = {}
for label in frequent_labels[:20]:
    n_sample = min(300, len(text_to_indices[label]) * (len(text_to_indices[label]) - 1) // 2)
    pairs = sample_same_label_pairs(label, n=n_sample)
    ia = torch.tensor([p[0] for p in pairs])
    ib = torch.tensor([p[1] for p in pairs])
    sims = (embeddings[ia] * embeddings[ib]).sum(dim=1).numpy()
    same_label_results[label] = sims.mean()
    diff = sims.mean() - sims_random.mean()
    print(f"{label:30s} {len(text_to_indices[label]):>10d} {sims.mean():>18.3f} {sims_random.mean():>16.3f} {diff:>+12.3f}")

## 3. Rango efectivo del espacio (vía SVD)

De las 128 dimensiones disponibles, ¿cuántas se usan realmente para diferenciar outfits? Un
rango efectivo muy bajo (participation ratio << 128) es otra forma de medir el mismo problema
de colapso/anisotropía, complementaria al histograma de la celda 1.

In [ ]:
X = embeddings.numpy()
X_centered = X - X.mean(axis=0, keepdims=True)
_, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

explained = (S ** 2) / (S ** 2).sum()
cum_explained = np.cumsum(explained)
eff_rank = (S.sum() ** 2) / (S ** 2).sum()  # participation ratio

print(f"Rango efectivo (participation ratio): {eff_rank:.1f} de 128 dimensiones posibles")
print(f"Varianza explicada por los primeros 2 componentes:  {cum_explained[1]*100:.1f}%")
print(f"Varianza explicada por los primeros 10 componentes: {cum_explained[9]*100:.1f}%")
print(f"Varianza explicada por los primeros 30 componentes: {cum_explained[29]*100:.1f}%")

plt.figure(figsize=(7,4))
plt.plot(np.arange(1, len(S)+1), cum_explained, marker="o", markersize=3)
plt.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="90% varianza")
plt.title("Varianza acumulada explicada por componente (SVD de outfit_emb)")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada explicada")
plt.legend()
plt.show()

## 4. Proyección 2D coloreada por etiqueta de estilo frecuente

Inspección visual directa: si hay algo de estructura por estilo, deberíamos ver *algo* de
agrupamiento por color aunque sea parcial (no esperamos clusters perfectamente separados con
solo 2 componentes de 128 -- esto es más una comprobación cualitativa rápida que una prueba
rigurosa).

In [ ]:
top_labels = frequent_labels[:8]
proj = X_centered @ Vt[:2].T  # (N, 2)

plt.figure(figsize=(9, 9))
colors = plt.cm.tab10(np.linspace(0, 1, len(top_labels)))

# fondo: todos los outfits en gris claro, para dar contexto de densidad
plt.scatter(proj[:, 0], proj[:, 1], s=4, alpha=0.05, color="gray")

for color, label in zip(colors, top_labels):
    idxs = text_to_indices[label]
    plt.scatter(proj[idxs, 0], proj[idxs, 1], s=10, alpha=0.6, color=color,
                label=f"{label} (n={len(idxs)})")

plt.legend(fontsize=8, loc="best")
plt.title("Proyección PCA 2D de outfit_emb, coloreada por etiqueta de texto")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

---
## Cómo decidir el siguiente paso con estos resultados

| Observación | Interpretación | Siguiente paso sugerido |
|---|---|---|
| Similitud aleatoria alta (>0.5) Y diferencia por etiqueta ≈ 0 | Colapso total: el espacio no lleva señal de estilo | El fine-tuning de la última capa es casi obligatorio, o incluso reconsiderar qué checkpoint/pooling usar |
| Similitud aleatoria baja/moderada Y diferencia por etiqueta claramente positiva (ej. +0.05 a +0.15) | Hay señal real, aunque débil | El resultado del notebook 2 es coherente con "señal débil pero real" -- el fine-tuning debería amplificarla, no crearla de la nada |
| Rango efectivo muy bajo (<15-20) | El cuello de botella es geométrico, no solo de entrenamiento | Fine-tuning de más capas (no solo la última), o reconsiderar la dimensión de salida |
| Rango efectivo razonable (>40-50) pero diferencia por etiqueta baja | El espacio tiene "sitio" pero no lo usa para estilo | El fine-tuning debería ayudar bastante, ya que hay grados de libertad disponibles para reorientar |

Con estos números ya tenemos evidencia objetiva (no solo la intuición de "seguramente el espacio
de compatibilidad no captura estilo") para decidir si el fine-tuning merece la pena y con qué
alcance.